# Embedding model comparison — Qwen3-0.6B vs Granite R2 (311M / 97M)

**Question:** Qwen3-Embedding-0.6B measured ~1.7 s/chunk on this laptop — too slow for the
experiment loop. Can a smaller model with the same `SentenceTransformer` API, long context
(10 KB chunks ≈ 3k tokens), multilingual + code coverage replace it *for experiments*?

**Candidates:** IBM Granite Embedding Multilingual R2 — 32k context, 200+ languages + code,
Apache 2.0, encoder-only (inherently faster per token than Qwen3's causal-LM design):
- `ibm-granite/granite-embedding-311m-multilingual-r2` (768 dims)
- `ibm-granite/granite-embedding-97m-multilingual-r2` (384 dims)

**Method** (nothing from the Qwen3 era was edited or removed — research history stays):
- `benchmark_embedding_models.py` — timed re-encode of the identical 510-chunk corpus per
  model, same batch_size=8, results appended to `.hrono/embedding-benchmark.json`.
- `test_embedding_models.py` — embeds each candidate into the `(hash, model)`-keyed
  `embeddings` table and runs the **frozen** clustering recipe imported from
  `build_clusters.py` (combined α=0.5 → UMAP-25 → HDBSCAN leaf → noise assignment),
  writing `clusters-<model>.json` (baseline `clusters.json` untouched).
- This notebook re-runs the comparison from those cached vectors and renders the results.

In [1]:
import json
import sqlite3
from pathlib import Path

import numpy as np
from sklearn.metrics import adjusted_rand_score

import build_clusters as bc

HRONO = Path(".hrono")
BASELINE = "Qwen/Qwen3-Embedding-0.6B"
CANDIDATES = [
    "ibm-granite/granite-embedding-311m-multilingual-r2",
    "ibm-granite/granite-embedding-97m-multilingual-r2",
]

## 1. Timing — embedding 510 chunks (~1 MB of text), batch 8

From the recorded benchmark runs (`.hrono/embedding-benchmark.json`). Timings are a fresh
re-encode of everything — no incremental skipping — so models are directly comparable.

In [2]:
runs = json.loads((HRONO / "embedding-benchmark.json").read_text())
run = runs[-1]  # most recent run; earlier runs are kept as history
m = run["machine"]
print(f"machine: {m['chip']}, {m['ram_gb']} GB RAM, torch {m['torch']} ({m['backend']})")
print(f"corpus:  {run['corpus_chunks']} unique-hash chunks, {run['corpus_kb']} KB, batch_size={run['batch_size']}\n")

header = f"{'model':<52} {'params':>7} {'dims':>5} {'ms/chunk':>9} {'chunks/s':>9} {'KB/s':>6} {'vs Qwen3':>9}"
print(header); print("-" * len(header))
base_ms = next(r["ms_per_chunk"] for r in run["results"] if r["model"] == BASELINE)
for r in run["results"]:
    if "error" in r:
        print(f"{r['model']:<52} FAILED: {r['error']}"); continue
    print(f"{r['model']:<52} {r['params_m']:>6}M {r['dims']:>5} {r['ms_per_chunk']:>9} "
          f"{r['chunks_per_s']:>9} {r['kb_per_s']:>6} {base_ms / r['ms_per_chunk']:>8.1f}x")

machine: Apple M1, 16 GB RAM, torch 2.13.0 (mps)
corpus:  510 unique-hash chunks, 1023 KB, batch_size=8

model                                                 params  dims  ms/chunk  chunks/s   KB/s  vs Qwen3
-------------------------------------------------------------------------------------------------------
Qwen/Qwen3-Embedding-0.6B                               596M  1024       835       1.2    2.4      1.0x
ibm-granite/granite-embedding-311m-multilingual-r2      312M   768       299      3.34    6.7      2.8x
ibm-granite/granite-embedding-97m-multilingual-r2        97M   384        75     13.29   26.7     11.1x


## 2. Clustering — does the frozen recipe still produce a sane tree?

Re-run from the cached vectors for each model (identical code path as the baseline:
`build_clusters.py` functions). Checks, per model:
- **known-group test** — webphone's 12 uk/es/en `translation.json` chunks must land in ONE cluster
- cluster count / size range (want the "small parents" leaf shape, sizes ~5-30)
- pre-assignment noise (Qwen3 leaf band on this corpus: 17-33%)
- coherence (CAVEAT: computed in each model's *own* space — NOT comparable across models)
- **ARI vs the Qwen3 baseline partition** (`clusters.json`)

In [3]:
def locales_hashes():
    db = sqlite3.connect(bc.DB_PATH)
    rows = db.execute("SELECT hash FROM chunks WHERE path LIKE '%locales%'").fetchall()
    db.close()
    return {r[0] for r in rows}


LOCALES = locales_hashes()
baseline_clusters = json.loads((HRONO / "clusters.json").read_text())
base_label_by_hash = {mm["hash"]: c["cluster_id"] for c in baseline_clusters for mm in c["members"]}


def compare(model_name):
    hashes, chunk_paths, content_vecs = bc.load_content_vectors(model_name)
    path_vecs = bc.embed_paths(chunk_paths, model_name, batch_size=8)
    combined = np.hstack([np.sqrt(1 - bc.ALPHA) * content_vecs, np.sqrt(bc.ALPHA) * path_vecs])
    labels, assigned, sim = bc.cluster(combined)
    clusters = bc.build_cluster_records(hashes, chunk_paths, content_vecs, labels, assigned, sim)

    label_by_hash = dict(zip(hashes, labels.tolist()))
    translation_clusters = {label_by_hash[h] for h in LOCALES if h in label_by_hash}
    common = [h for h in hashes if h in base_label_by_hash]
    sizes = sorted((c["size"] for c in clusters), reverse=True)
    return {
        "model": model_name,
        "clusters": len(clusters),
        "size_range": f"{sizes[-1]}-{sizes[0]} (median {int(np.median(sizes))})",
        "noise_pct": round(100 * float(assigned.mean())),
        "coherence_median": float(np.median([c["stats"]["coherence"] for c in clusters])),
        "known_group": "PASS" if len(translation_clusters) == 1 else f"FAIL ({len(translation_clusters)} clusters)",
        "ari_vs_baseline": round(adjusted_rand_score(
            [base_label_by_hash[h] for h in common], [label_by_hash[h] for h in common]), 3),
    }


results = [compare(name) for name in [BASELINE] + CANDIDATES]

header = (f"{'model':<52} {'clusters':>8} {'sizes':>20} {'noise%':>7} "
          f"{'coher.':>7} {'known-group':>12} {'ARI':>6}")
print(header); print("-" * len(header))
for r in results:
    print(f"{r['model']:<52} {r['clusters']:>8} {r['size_range']:>20} {r['noise_pct']:>7} "
          f"{r['coherence_median']:>7.3f} {r['known_group']:>12} {r['ari_vs_baseline']:>6}")
print("\nARI context: two independent Qwen3 cold rebuilds agree at ~0.81 (research.ipynb) — that's the")
print("realistic ceiling; the order-bias experiment showed 'different but equally good' partitions at 0.715.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

model load: 5.6s


embedded 482 distinct paths


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP: 2048 -> 25 dims
HDBSCAN leaf: 37 clusters, 85 noise chunks to assign


...

model load: 5.0s


embedded 482 distinct paths


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP: 768 -> 25 dims
HDBSCAN leaf: 38 clusters, 111 noise chunks to assign
model                                                clusters                sizes  noise%  coher.  known-group    ARI
----------------------------------------------------------------------------------------------------------------------
Qwen/Qwen3-Embedding-0.6B                                  37     5-29 (median 10)      17   0.651         PASS    1.0
ibm-granite/granite-embedding-311m-multilingual-r2         41     6-25 (median 12)      20   0.898         PASS  0.546
ibm-granite/granite-embedding-97m-multilingual-r2          38     5-32 (median 12)      22   0.873         PASS  0.448

ARI context: two independent Qwen3 cold rebuilds agree at ~0.81 (research.ipynb) — that's the
realistic ceiling; the order-bias experiment showed 'different but equally good' partitions at 0.715.


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


## 3. Model-independent quality check: path coherence

ARI only says the partitions *differ*; it cannot say which is better, and embedding-space
metrics (coherence) are incomparable across models. Path coherence is model-independent:
size-weighted share of cluster members whose 3-segment path prefix matches the cluster's
dominant prefix (same metric as research.ipynb's alpha sweep). Computed from the saved
cluster files, post-assignment (no noise class).

In [4]:
FILES = {
    "Qwen/Qwen3-Embedding-0.6B": "clusters.json",
    "ibm-granite/granite-embedding-311m-multilingual-r2": "clusters-granite-embedding-311m-multilingual-r2.json",
    "ibm-granite/granite-embedding-97m-multilingual-r2": "clusters-granite-embedding-97m-multilingual-r2.json",
}
for model_name, fname in FILES.items():
    clusters = json.loads((HRONO / fname).read_text())
    agree = sum(c["stats"]["dominant_prefixes"][0]["count"] for c in clusters)
    total = sum(c["size"] for c in clusters)
    print(f"{model_name:<52} path_coherence = {agree / total:.3f}")
print("\nA statistical tie: all three partitions align with directory structure equally well —")
print("the ARI differences are different-but-equally-path-consistent boundaries, not quality loss.")

Qwen/Qwen3-Embedding-0.6B                            path_coherence = 0.947
ibm-granite/granite-embedding-311m-multilingual-r2   path_coherence = 0.949
ibm-granite/granite-embedding-97m-multilingual-r2    path_coherence = 0.939

A statistical tie: all three partitions align with directory structure equally well —
the ARI differences are different-but-equally-path-consistent boundaries, not quality loss.


## Conclusion

**Decision: `granite-embedding-97m-multilingual-r2` is the experiment-loop model;
Qwen3-Embedding-0.6B stays the final-run model.** (`embedding.toml` still points at Qwen3 —
the test scripts pass the model name explicitly; flipping the config is a deliberate act.)

- **Speed:** 97M is ~11x faster than Qwen3 under identical conditions (75 vs 835 ms/chunk —
  note Qwen3's original ~1700 ms/chunk was measured under different load; the controlled
  number is 835). 311M at ~3x is not worth it: it buys ARI 0.55 vs 0.45, both far below the
  0.81 rebuild-agreement ceiling, so neither reproduces Qwen3's partition anyway.
- **Quality:** path coherence — the one model-independent metric — is a tie (0.947 / 0.949 / 0.939), and both Granites PASS the known-group test and produce the right leaf shape
  (sizes ~5-32, noise 20-22% — inside Qwen3's 17-33% band). The ARI ~0.45-0.55 vs baseline
  means a *different but structurally equivalent* tree — same class of difference as the
  order-bias experiment's "different but equally good" (0.715).
- **Implication for experiments:** results obtained on 97M embeddings (clustering behavior,
  update mechanics, summarization inputs) transfer as *mechanisms*, but concrete cluster
  memberships will differ from Qwen3 runs — don't compare partitions across models, and
  re-validate any partition-sensitive finding on the final-run model before freezing it.
- Coherence medians (0.65 Qwen3 / 0.90 / 0.87 Granite) are each computed in the model's own
  embedding space — absolute values are NOT comparable across models (Qwen3's space simply runs
  lower cosine similarities overall), so this row says nothing about relative quality.
- Vectors for all three models coexist in the `(hash, model)`-keyed `embeddings` table;
  384-dim vectors also make experiment-scale similarity scans ~2.7x cheaper than 1024-dim.